In [387]:
import pandas as pd

In [388]:
""" !wget https://raw.githubusercontent.com/alexeygrigorev/datasets/master/course_lead_scoring.csv """

' !wget https://raw.githubusercontent.com/alexeygrigorev/datasets/master/course_lead_scoring.csv '

In [389]:
df = pd.read_csv('./data/course_lead_scoring.csv')
df

,lead_source,industry,number_of_courses_viewed,annual_income,employment_status,location,interaction_count,lead_score,converted
0,paid_ads,NaN,1,79450.0,unemployed,south_america,4,0.94,1
1,social_media,retail,1,46992.0,employed,south_america,1,0.80,0
2,events,healthcare,5,78796.0,unemployed,australia,3,0.69,1
3,paid_ads,retail,2,83843.0,NaN,australia,1,0.87,0
4,referral,education,3,85012.0,self_employed,europe,3,0.62,1
...,...,...,...,...,...,...,...,...,...
1457,referral,manufacturing,1,NaN,self_employed,north_america,4,0.53,1
1458,referral,technology,3,65259.0,student,europe,2,0.24,1
1459,paid_ads,technology,1,45688.0,student,north_america,3,0.02,1
1460,referral,NaN,5,71016.0,self_employed,north_america,0,0.25,1


In [390]:
categorical_columns = list(df.columns[df.dtypes == 'object'])
numerical_columns = list(df.columns[(df.dtypes != 'object') & (df.columns != 'converted')])

for col in categorical_columns:
    df[col] = df[col].fillna('NA')

for col in numerical_columns:
    df[col] = df[col].fillna(0.0)

In [391]:
df.isnull().sum()

lead_source                 0
industry                    0
number_of_courses_viewed    0
annual_income               0
employment_status           0
location                    0
interaction_count           0
lead_score                  0
converted                   0
dtype: int64

In [392]:
df['industry'].mode()

0    retail
Name: industry, dtype: object

In [393]:
df_numerical = df[numerical_columns]
df_numerical.corr().round(2)

,number_of_courses_viewed,annual_income,interaction_count,lead_score
number_of_courses_viewed,1.00,0.01,-0.02,-0.00
annual_income,0.01,1.00,0.03,0.02
interaction_count,-0.02,0.03,1.00,0.01
lead_score,-0.00,0.02,0.01,1.00


In [394]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import mutual_info_score
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression

In [395]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=42)

In [396]:
df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

y_train = df_train['converted'].values
y_val = df_val['converted'].values
y_test = df_test['converted'].values

del df_train['converted']
del df_val['converted']
del df_test['converted']

In [397]:
df_full_train = df_full_train.reset_index(drop=True)

In [398]:
def mutual_info(series):
    return mutual_info_score(series, df_full_train['converted'])

mi = df_full_train[categorical_columns].apply(mutual_info)
mi.sort_values(ascending=False).round(2)

lead_source          0.03
employment_status    0.01
industry             0.01
location             0.00
dtype: float64

In [399]:
dv = DictVectorizer(sparse=False)

train_dict = df_train[categorical_columns + numerical_columns].to_dict(orient='records')
X_train = dv.fit_transform(train_dict)

val_dict = df_val[categorical_columns + numerical_columns].to_dict(orient='records')
X_val = dv.transform(val_dict)

model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
model.fit(X_train, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,42
,solver,'liblinear'
,max_iter,1000
,multi_class,'deprecated'


In [1]:
X_train

NameError: name 'X_train' is not defined

In [400]:
y_pred = model.predict_proba(X_val)[:, 1]
decision = (y_pred >= 0.5)
accuracy_default = (decision == y_test).mean()
accuracy_default

np.float64(0.6382252559726962)

In [401]:
features = ['industry', 'employment_status', 'lead_score']
accuracy_dict = {}

for feature in features:
    all_features = categorical_columns + numerical_columns
    all_features = [f for f in all_features if f != feature]

    train_dict = df_train[all_features].to_dict(orient='records')
    X_train = dv.fit_transform(train_dict)

    val_dict = df_val[all_features].to_dict(orient='records')
    X_val = dv.transform(val_dict)

    model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
    model.fit(X_train, y_train)

    y_pred = model.predict_proba(X_val)[:, 1]
    decision = (y_pred >= 0.5)
    accuracy = (decision == y_test).mean()

    accuracy_dict[feature] = accuracy

accuracy_dict

{'industry': np.float64(0.6382252559726962),
 'employment_status': np.float64(0.6279863481228669),
 'lead_score': np.float64(0.6313993174061433)}

In [402]:
diff_dict = {k: abs(v - accuracy_default) for k, v in accuracy_dict.items()}
diff_dict

{'industry': np.float64(0.0),
 'employment_status': np.float64(0.010238907849829282),
 'lead_score': np.float64(0.0068259385665528916)}

In [403]:
params = [0.01, 0.1, 1, 10, 100]
accuracy_params_dict = {}

for c in params:
    train_dict = df_train[categorical_columns + numerical_columns].to_dict(orient='records')
    X_train = dv.fit_transform(train_dict)

    val_dict = df_val[categorical_columns + numerical_columns].to_dict(orient='records')
    X_val = dv.transform(val_dict)

    model = LogisticRegression(solver='liblinear', C=c, max_iter=1000, random_state=42)
    model.fit(X_train, y_train)

    y_pred = model.predict_proba(X_val)[:, 1]
    decision = (y_pred >= 0.5)
    accuracy = (decision == y_test).mean()

    accuracy_params_dict[c] = accuracy.round(3)

accuracy_params_dict

{0.01: np.float64(0.638),
 0.1: np.float64(0.638),
 1: np.float64(0.638),
 10: np.float64(0.638),
 100: np.float64(0.638)}